## Project 4 Machine Learning

So in this assignment I have reused my notebook from assignment 2. I started with the Elhub api to retrive hourly production data for year 2022 to 2024. Then setting up the same way I did earlier. Setting up Cassandra using Spark and then check connection to MongoDB, and later upload to MongoDB. I have to reuse the Database by adding new tables and then retrieve hourly data for all price areas. 

I start with the Jupyter Notebook, because the workload are less then the Streamlit application.

# AI Usage
I integrated AI-powered development tools, including GitHub Copilot and Google Gemini 3, into my workflow. These served as both a programming partner for code generation and problem-solving and a sparring partner for refactoring, reviewing alternative approaches, and enhancing code quality and efficiency.




In [33]:
# MONGODB
import toml
from pymongo import MongoClient
from pymongo.server_api import ServerApi
# Load secrets
secrets = toml.load(".streamlit/secrets.toml")
uri = secrets["MONGO"]["uri"]

# Create a new client and connect to the server
client = MongoClient(uri, server_api=ServerApi('1'))

# Send a ping to confirm a successful connection
try:
    client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(e)

Pinged your deployment. You successfully connected to MongoDB!


In [34]:
#Spark and Cassandra
import os
# --- Java + Hadoop setup ---
os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.16.8-hotspot"
os.environ["HADOOP_HOME"] = r"C:\Hadoop\hadoop-3.3.1"
os.environ["PYSPARK_HADOOP_VERSION"] = "without"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.path.join(os.environ["HADOOP_HOME"], "bin") + ";" + os.environ["PATH"]


In [35]:
import os
import requests
import json
import pandas as pd
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta 

def fetch_data_for_period(api_name, start_year, end_year):
    """
    Henter data fra Elhub API for angitt periode og API-navn.
    Inkluderer robust fletting og typekontroll.
    """
    start_date = datetime(start_year, 1, 1)
    # Henter data opp til slutten av end_year
    end_date = datetime(end_year, 12, 31) + timedelta(days=1) 
    
    merged_data = None
    current_date = start_date

    # Definerer data_key eksplisitt
    if 'PRODUCTION' in api_name:
        data_key = 'productionPerGroupMbaHour'
    elif 'CONSUMPTION' in api_name:
        data_key = 'consumptionPerGroupMbaHour'
    else:
        raise ValueError(f"Ukjent API-navn: {api_name}")

    print(f"--- Starter henting for {api_name} ({start_year}-{end_year}) ---")

    while current_date < end_date:
        next_month = current_date + relativedelta(months=1)
        # Sørger for at vi ikke henter data etter slutten av end_year
        month_end_inclusive = min(next_month, end_date) - timedelta(seconds=1)

        start_formatted = current_date.strftime('%Y-%m-%dT00:00:00+02:00').replace(':', '%3A').replace('+', '%2B')
        end_formatted = month_end_inclusive.strftime('%Y-%m-%dT23:59:59+02:00').replace(':', '%3A').replace('+', '%2B')

        url = (
            f"https://api.elhub.no/energy-data/v0/price-areas"
            f"?dataset={api_name}"
            f"&startDate={start_formatted}&endDate={end_formatted}"
        )

        print(f"Henter data for {current_date.strftime('%Y-%m')}...")

        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()

            if data and 'data' in data:
                if merged_data is None:
                    merged_data = data
                else:
                    # KORRIGERT FLETTE-LOGIKK:
                    for new_area in data['data']:
                        found = False
                        for existing_area in merged_data['data']:
                            if existing_area['attributes']['name'] == new_area['attributes']['name']:
                                
                                # Sjekk for å sikre at begge er lister FØR vi kaller extend()
                                existing_list = existing_area['attributes'].get(data_key)
                                new_list = new_area['attributes'].get(data_key)
                                
                                if isinstance(existing_list, list) and isinstance(new_list, list):
                                    existing_list.extend(new_list)
                                elif new_list is not None and isinstance(new_list, list):
                                    # Hvis den eksisterende av en eller annen grunn er None/korrupt, men den nye er gyldig, bruk den nye.
                                    existing_area['attributes'][data_key] = new_list
                                else:
                                    # Hvis dataen i den nye måneden er en streng/feil, ignoreres den.
                                    pass 
                                
                                found = True
                                break
                        if not found:
                            merged_data['data'].append(new_area)
            
        except Exception as error:
            print(f"Feil ved henting av data for {current_date.strftime('%Y-%m')}: {error}")
            
        current_date = next_month
    
    print("--- Henting fullført ---")
    return merged_data

In [36]:
# Kjøres først: Tilkobling til Cassandra
# Import Cassandra cluster connection
from cassandra.cluster import Cluster
# Connect to Cassandra cluster
cluster = Cluster(['localhost'], port=9042)
session = cluster.connect()
print("Connected to Cassandra cluster:", cluster)

# --- FIKS FOR KEYSPACE ---
keyspace = "energy_data" 
cql_create_keyspace = f"""
    CREATE KEYSPACE IF NOT EXISTS {keyspace}
    WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': '1'}}
"""
session.execute(cql_create_keyspace)
print(f"✅ Keyspace '{keyspace}' er bekreftet/opprettet.")
session.set_keyspace(keyspace)
print(f"✅ Cassandra-økten bruker nå Keyspace: {keyspace}")
consumption_table = "consumption_per_group"

Connected to Cassandra cluster: <cassandra.cluster.Cluster object at 0x00000184B2191C90>
✅ Keyspace 'energy_data' er bekreftet/opprettet.
✅ Cassandra-økten bruker nå Keyspace: energy_data


In [37]:
# Keyspace og consumption_table er allerede definert
# keyspace = "energy_data"
# consumption_table = "consumption_per_group"

cql_create_consumption_table = f"""
    CREATE TABLE IF NOT EXISTS {keyspace}.{consumption_table} (
        priceArea text,
        consumptionGroup text,
        startTime timestamp,
        endTime timestamp,
        quantityKwh double,
        lastUpdatedTime timestamp,
        PRIMARY KEY ((priceArea, consumptionGroup), startTime)
    );
"""
session.execute(cql_create_consumption_table)
print(f"✅ Ny tabell '{keyspace}.{consumption_table}' er bekreftet/opprettet i Cassandra.")

✅ Ny tabell 'energy_data.consumption_per_group' er bekreftet/opprettet i Cassandra.


In [ ]:
from pyspark.sql.functions import to_timestamp

# --- Steg 4: Last inn data i Cassandra (Fyll tabellen) ---

# 0. Klargjør Pandas DF ved å konvertere datetime-kolonner til strenger 
# Dette er kritisk for å tvinge riktig typeinferens i Spark.
temp_df = consumption_21_24_df.copy()
datetime_columns = ['startTime', 'endTime', 'lastUpdatedTime']

for col_name in datetime_columns:
    if col_name in temp_df.columns:
        # Konverterer Pandas datetime objekter til 'YYYY-MM-DD HH:MM:SS' strengformat
        # Bruker .dt.strftime kun hvis kolonnen er datetime, ellers feiler det.
        if pd.api.types.is_datetime64_any_dtype(temp_df[col_name]):
             temp_df[col_name] = temp_df[col_name].dt.strftime('%Y-%m-%d %H:%M:%S')

# 1. Konverter KORRIGERT Pandas DF til Spark DF
cons_spark_df = spark.createDataFrame(temp_df)

# 2. Formater datatypene i Spark (nå som de er StringType)
# Vi må eksplisitt angi formatet vi brukte ovenfor.
time_format = "yyyy-MM-dd HH:mm:ss"

cons_spark_df = cons_spark_df.withColumn("startTime", to_timestamp("startTime", time_format)) \
                           .withColumn("endTime", to_timestamp("endTime", time_format)) \
                           .withColumn("lastUpdatedTime", to_timestamp("lastUpdatedTime", time_format))

# 3. Lagre i tabellen
cons_spark_df.write.format("org.apache.spark.sql.cassandra") \
    .options(table=consumption_table, keyspace=keyspace) \
    .mode("overwrite") \
    .save()
print(f"✅ Forbruksdata (2021-2024) er nå lastet inn i Cassandra-tabell '{consumption_table}'.")

In [38]:
# Import Spark session for distributed data processing
from pyspark.sql import SparkSession

# Create or get existing Spark session with Cassandra connector configuration
spark = SparkSession.builder.appName('SparkCassandraApp').\
    config('spark.jars.packages', 'com.datastax.spark:spark-cassandra-connector_2.12:3.5.1').\
    config('spark.cassandra.connection.host', 'localhost').\
    config('spark.sql.extensions', 'com.datastax.spark.connector.CassandraSparkExtensions').\
    config('spark.sql.catalog.mycatalog', 'com.datastax.spark.connector.datasource.CassandraCatalog').\
    config('spark.cassandra.connection.port', '9042').getOrCreate()
# Some warnings are to be expected.
# If running this cell does not give any output after ~30 seconds, there is likely an error in the configuration (JAVA_HOME, HADOOP_HOME, etc.).

In [ ]:
# Hjelpefunksjon: Konverterer Elhub JSON til Pandas DataFrame
def json_to_df(data, data_key):
    """
    Konverterer Elhub 'price-areas' JSON til flat pandas DataFrame.
    data_key f.eks. 'productionPerGroupMbaHour' eller 'consumptionPerGroupMbaHour'.
    Hver post får kolonnen 'priceArea' fra attributes.name.
    """
    if not data or 'data' not in data:
        return pd.DataFrame()

    rows = []
    for area in data.get('data', []):
        attrs = area.get('attributes', {}) or {}
        price_area = attrs.get('name')
        items = attrs.get(data_key) or []
        # Hvis items ikke er en liste, hopp over
        if not isinstance(items, list):
            continue
        for rec in items:
            r = rec.copy()
            r['priceArea'] = price_area
            rows.append(r)

    df = pd.DataFrame(rows)
    # Hvis kolonnen 'group' finnes (som i Elhub), behold den slik at eksisterende kode kan rename() den videre.
    return df

# --- Hent og forbered Produksjonsdata (2022-2024) ---
production_22_24_json = fetch_data_for_period(
    api_name='PRODUCTION_PER_GROUP_MBA_HOUR',
    start_year=2022,
    end_year=2024
)
production_22_24_df = json_to_df(production_22_24_json, 'productionPerGroupMbaHour')
production_22_24_df = production_22_24_df.rename(columns={'group': 'productionGroup'})
print(f"Produksjonsdata (2022-2024) klar: {len(production_22_24_df):,} rader")

# --- Hent og forbered Forbruksdata (2021-2024) ---
consumption_21_24_json = fetch_data_for_period(
    api_name='CONSUMPTION_PER_GROUP_MBA_HOUR',
    start_year=2021,
    end_year=2024
)
# Denne linjen definerer DataFrame-en du manglet!
consumption_21_24_df = json_to_df(consumption_21_24_json, 'consumptionPerGroupMbaHour')
consumption_21_24_df = consumption_21_24_df.rename(columns={'group': 'consumptionGroup'})
print(f"Forbruksdata (2021-2024) klar: {len(consumption_21_24_df):,} rader")

# Lagre forbruks-JSON i mappen du spesifiserte
if consumption_21_24_json:
    if not os.path.isdir('./JSON_data'):
        os.makedirs('./JSON_data', exist_ok=True)
    with open('./JSON_data/consumption_data_2021_2024.json', 'w', encoding='utf-8') as f:
        json.dump(consumption_21_24_json, f, indent=4)
    print("✅ Consumption JSON data saved.")

--- Starter henting for PRODUCTION_PER_GROUP_MBA_HOUR (2022-2024) ---
Henter data for 2022-01...
Henter data for 2022-02...
Henter data for 2022-03...
Henter data for 2022-04...
Henter data for 2022-05...
Henter data for 2022-06...
Henter data for 2022-07...
Henter data for 2022-08...
Henter data for 2022-09...
Henter data for 2022-10...
Henter data for 2022-11...
Henter data for 2022-12...
Henter data for 2023-01...
Henter data for 2023-02...
Henter data for 2023-03...
Henter data for 2023-04...
Henter data for 2023-05...
Henter data for 2023-06...
Henter data for 2023-07...
Henter data for 2023-08...
Henter data for 2023-09...
Henter data for 2023-10...
Henter data for 2023-11...
Henter data for 2023-12...
Henter data for 2024-01...
Henter data for 2024-02...
Henter data for 2024-03...
Henter data for 2024-04...
Henter data for 2024-05...
Henter data for 2024-06...
Henter data for 2024-07...
Henter data for 2024-08...
Henter data for 2024-09...
Henter data for 2024-10...
Henter data 

C:\Users\Erik\AppData\Local\Temp\ipykernel_10308\1968998906.py:89: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Erik\AppData\Local\Temp\ipykernel_10308\1968998906.py:89: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Erik\AppData\Local\Temp\ipykernel_10308\1968998906.py:89: FutureWarning: In a future version of pandas, parsing

Produksjonsdata (2022-2024) klar: 657,600 rader
--- Starter henting for CONSUMPTION_PER_GROUP_MBA_HOUR (2021-2024) ---
Henter data for 2021-01...
Henter data for 2021-02...
Henter data for 2021-03...
Henter data for 2021-04...
Henter data for 2021-05...
Henter data for 2021-06...
Henter data for 2021-07...
Henter data for 2021-08...
Henter data for 2021-09...
Henter data for 2021-10...
Henter data for 2021-11...
Henter data for 2021-12...
Henter data for 2022-01...
Henter data for 2022-02...
Henter data for 2022-03...
Henter data for 2022-04...
Henter data for 2022-05...
Henter data for 2022-06...
Henter data for 2022-07...
Henter data for 2022-08...
Henter data for 2022-09...
Henter data for 2022-10...
Henter data for 2022-11...
Henter data for 2022-12...
Henter data for 2023-01...
Henter data for 2023-02...
Henter data for 2023-03...
Henter data for 2023-04...
Henter data for 2023-05...
Henter data for 2023-06...
Henter data for 2023-07...
Henter data for 2023-08...
Henter data for 2

C:\Users\Erik\AppData\Local\Temp\ipykernel_10308\1968998906.py:89: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Erik\AppData\Local\Temp\ipykernel_10308\1968998906.py:89: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df[col] = pd.to_datetime(df[col], errors='coerce')
C:\Users\Erik\AppData\Local\Temp\ipykernel_10308\1968998906.py:89: FutureWarning: In a future version of pandas, parsing

Forbruksdata (2021-2024) klar: 876,600 rader
✅ Consumption JSON data saved.


In [40]:
from pyspark.sql.functions import to_timestamp

# MERK: Krever at 'consumption_21_24_df' Pandas DataFrame er definert og fylt med 2021-2024 data.

# 1. Konverter Pandas DF til Spark DF for forbruk
cons_spark_df = spark.createDataFrame(consumption_21_24_df)

# 2. Formater datatypene (Viktig for Cassandra)
cons_spark_df = cons_spark_df.withColumn(
    "startTime", 
    to_timestamp("startTime")
).withColumn(
    "endTime", 
    to_timestamp("endTime")
).withColumn(
    "lastUpdatedTime", 
    to_timestamp("lastUpdatedTime")
)

# 3. Lagre i den nye tabellen
cons_spark_df.write.format("org.apache.spark.sql.cassandra") \
    .options(table=consumption_table, keyspace=keyspace) \
    .mode("overwrite") \
    .save()
print(f"✅ Forbruksdata (2021-2024) lastet inn i Cassandra-tabell '{consumption_table}'.")

{"ts": "2025-11-22 17:16:52.159", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"to_timestamp(startTime)\" due to data type mismatch: The first parameter requires the (\"STRING\" or \"DATE\" or \"TIMESTAMP\" or \"TIMESTAMP_NTZ\" or \"NUMERIC\") type, however \"startTime\" has the type \"STRUCT<>\". SQLSTATE: 42K09", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "to_timestamp", "errorClass": "DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o84.withColumn.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"to_timestamp(startTime)\" due to data type mismatch: The first parameter requires the (\"STRING\" or \"DATE\" or \"TIMESTAMP\" or \"TIMESTAMP_NTZ\" or \"NUMERIC\") type, however \"startTime\" has the type 

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "to_timestamp(startTime)" due to data type mismatch: The first parameter requires the ("STRING" or "DATE" or "TIMESTAMP" or "TIMESTAMP_NTZ" or "NUMERIC") type, however "startTime" has the type "STRUCT<>". SQLSTATE: 42K09;
'Project [consumptionGroup#0, endTime#1, lastUpdatedTime#2, meteringPointCount#3L, priceArea#4, quantityKwh#5, to_timestamp(startTime#6, None, TimestampType, Some(Europe/Oslo), true) AS startTime#7]
+- LogicalRDD [consumptionGroup#0, endTime#1, lastUpdatedTime#2, meteringPointCount#3L, priceArea#4, quantityKwh#5, startTime#6], false


In [ ]:
# Keyspace og consumption_table er definert i en tidligere celle (energy_data, consumption_per_group)

cql_create_consumption_table = f"""
    CREATE TABLE IF NOT EXISTS {keyspace}.{consumption_table} (
        priceArea text,
        consumptionGroup text,
        startTime timestamp,
        endTime timestamp,
        quantityKwh double,
        lastUpdatedTime timestamp,
        PRIMARY KEY ((priceArea, consumptionGroup), startTime)
    );
"""
session.execute(cql_create_consumption_table)
print(f"✅ Tabell '{consumption_table}' er bekreftet/opprettet.")

In [ ]:
from pyspark.sql.functions import to_timestamp

# 1. Konverter Pandas DF til Spark DF
cons_spark_df = spark.createDataFrame(consumption_21_24_df)

# 2. Formater datatypene for Cassandra
cons_spark_df = cons_spark_df.withColumn("startTime", to_timestamp("startTime")) \
                           .withColumn("endTime", to_timestamp("endTime")) \
                           .withColumn("lastUpdatedTime", to_timestamp("lastUpdatedTime"))

# 3. Lagre i tabellen
cons_spark_df.write.format("org.apache.spark.sql.cassandra") \
    .options(table=consumption_table, keyspace=keyspace) \
    .mode("overwrite") \
    .save()
print(f"✅ Forbruksdata (2021-2024) er nå lastet inn i Cassandra-tabell '{consumption_table}'.")

Py4JJavaError: An error occurred while calling o55.load.
: java.lang.NoSuchMethodError: 'scala.collection.convert.Decorators$AsScala scala.jdk.CollectionConverters$.mapAsScalaMapConverter(java.util.Map)'
	at org.apache.spark.sql.cassandra.DefaultSource.getTable(DefaultSource.scala:46)
	at org.apache.spark.sql.cassandra.DefaultSource.inferSchema(DefaultSource.scala:67)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceV2Utils$.getTableFromProvider(DataSourceV2Utils.scala:96)
	at org.apache.spark.sql.execution.datasources.v2.DataSourceV2Utils$.loadV2Source(DataSourceV2Utils.scala:147)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.$anonfun$applyOrElse$1(ResolveDataSource.scala:60)
	at scala.Option.flatMap(Option.scala:283)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataSource.scala:58)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource$$anonfun$apply$1.applyOrElse(ResolveDataS

In [ ]:
# --- Last inn eksisterende værdata (df_weather) ---
import pandas as pd

# 🚨 VIKTIG: Erstatt 'weather_data_master.parquet' med ditt faktiske filnavn 
# og riktig sti der du lagret værdataen fra Oppgave 3.

try:
    # Anbefalt: Last inn fra Parquet (raskere og bedre format)
    df_weather = pd.read_parquet('./data/weather_data_master.parquet')
except FileNotFoundError:
    try:
        # Alternativ: Last inn fra CSV hvis du lagret som det
        df_weather = pd.read_csv('./data/weather_data_master.csv')
    except FileNotFoundError:
        print("🛑 FEIL: Finner ikke værdatafilen. Sjekk filnavn og sti.")
        # Opprett en tom DataFrame for å unngå feil i de neste stegene
        df_weather = pd.DataFrame() 

if not df_weather.empty:
    # Sikre at 'time'-kolonnen er en datetime-indeks (Viktig for sammenslåing)
    if 'time' in df_weather.columns:
        df_weather['time'] = pd.to_datetime(df_weather['time'])
        df_weather = df_weather.set_index('time')
    elif not isinstance(df_weather.index, pd.DatetimeIndex):
        # Hvis indeksen ikke er navngitt, men er datetime
        df_weather.index = pd.to_datetime(df_weather.index)
        df_weather.index.name = 'time'

    # Rens opp i datatypene og fjern duplikater
    df_weather = df_weather.sort_index()
    df_weather = df_weather[~df_weather.index.duplicated(keep='first')]

    print(f"✅ df_weather DataFrame lastet og klargjort: {len(df_weather):,} rader.")
    print(f"Kolonner: {df_weather.columns.tolist()}")

In [ ]:
# --- Steg 2: Slå sammen med Værdata (Master DataFrame) ---
# Antar at df_weather er lastet inn og indeksert på tid

# Filtrer værdataen til relevante kolonner
weather_for_merge = df_weather[['temperature_2m (°C)', 'precipitation_amount (mm)']].copy()
weather_for_merge.index.name = 'time' 

# Slå sammen Forbruk og Værdata (inner join)
master_df = consumption_df.merge(
    weather_for_merge, 
    left_index=True, 
    right_index=True, 
    how='inner'
)
master_df = master_df.sort_index().dropna() 

print(f"✅ Master DataFrame (Forbruk + Vær) klar: {len(master_df):,} rader.")
print("\nMaster DataFrame (Første 5 rader):")
print(master_df.head())

In [ ]:
# Legg til tidsbaserte funksjoner
master_df['year'] = master_df.index.year
master_df['month'] = master_df.index.month
master_df['dayofweek'] = master_df.index.dayofweek # Mandag=0, Søndag=6
master_df['hour'] = master_df.index.hour

# Legg til en boolsk variabel for helg
master_df['is_weekend'] = (master_df['dayofweek'] >= 5).astype(int)

# 🚨 Viktig: Lag en lag/forskyvning (shift) av temperaturen 
# Strømforbruk korrelerer ofte med temperaturen i timen før/første, ikke bare i sanntid.
master_df['temp_lag1'] = master_df['temperature_2m (°C)'].shift(1)

# Fjern rader med NaN etter lag-operasjonen
master_df = master_df.dropna()

print(f"Master DataFrame etter Feature Engineering: {len(master_df):,} rader.")
print("\nMaster DataFrame (Siste 5 rader med nye features):")
print(master_df.tail())

In [ ]:
# --- Steg 4: Definisjon av Features og Target ---
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Kolonnen vi ønsker å forutsi
TARGET = 'consumption' 

# Kolonnene vi bruker for å forutsi TARGET (uavhengige variabler)
FEATURES = [
    'temperature_2m (°C)', 
    'precipitation_amount (mm)',
    'temp_lag1', # Forskyvet temperatur er en sterk prediktor
    'year', 
    'month', 
    'dayofweek', 
    'hour', 
    'is_weekend'
]

# Sjekk at alle FEATURES eksisterer i DataFrame
missing_features = [f for f in FEATURES if f not in master_df.columns]
if missing_features:
    raise ValueError(f"Følgende features mangler i master_df: {missing_features}")


X = master_df[FEATURES]
y = master_df[TARGET]

# --- Steg 5: Tog/Test Splitting ---
# Vi bruker typisk de siste 10% av dataene som testsett for tidsrekker
train_size = int(len(master_df) * 0.9)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f"Treningssett: {len(X_train)} rader, Testsett: {len(X_test)} rader.")


# --- Steg 6: Modelltrening (Random Forest) ---
print("\nStarter modelltrening (Random Forest Regressor)...")
# Bruk et moderat antall estimatere for å balansere ytelse og hastighet
rfr_model = RandomForestRegressor(
    n_estimators=100, 
    max_depth=15, # Begrens dybden litt
    random_state=42, 
    n_jobs=-1 # Bruk alle kjerner
)

rfr_model.fit(X_train, y_train)
print("✅ Modelltrening fullført.")


# --- Steg 7: Evaluering ---
y_pred = rfr_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- Evaluering av Modellen ---")
print(f"Mean Absolute Error (MAE): {mae:,.2f} kWh (Gjennomsnittlig avvik i prognose)")
print(f"R-squared (R²): {r2:.4f} (Forklart varians, mål på modellkvalitet)")
print("------------------------------")

In [ ]:
import joblib
# --- Steg 8: Lagre Modellen og Features ---
MODEL_PATH = './rfr_consumption_model.joblib'
FEATURES_PATH = './rfr_consumption_features.joblib'

# Lagre selve modellen
joblib.dump(rfr_model, MODEL_PATH)

# Lagre listen over features som modellen ble trent med (Viktig for Streamlit!)
joblib.dump(FEATURES, FEATURES_PATH)

print(f"✅ Modell lagret til: {MODEL_PATH}")
print(f"✅ Features lagret til: {FEATURES_PATH}")